# Module 3: Highest-Change District — Year-Wise Analysis

**Task 3**: Identify the district with highest change and perform year-wise analysis (2016→2025) to determine whether changes are gradual or abrupt.

**Deliverables**:
- Ranked district table
- Annual time-series line plots (2016–2025, 10 data points per class)
- Year-over-year change rate charts
- Gradual vs. abrupt classification with statistical evidence
- Narrative interpretation

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

from config import (
    STATES, YEARS, ALL_YEARS, LULC_CLASSES, LULC_COLORS, NUM_CLASSES,
    GEE_EXPORTS_DIR, PROCESSED_DIR, FIGURES_DIR,
    TIER1_SCALE, TIER3_SCALE, PIXEL_AREA_KM2,
)
from scripts.lulc_utils import (
    parse_histogram, histogram_to_area_df,
    save_composition_csv,
)
from scripts.visualization import (
    plot_annual_timeseries, save_figure,
)

print('Imports successful!')

## 3.1: Load District Rankings from Module 2

In [ ]:
# Load district change rankings
highest_change_districts = {}

for state_key, state_cfg in STATES.items():
    filepath = PROCESSED_DIR / 'transition_matrices' / f'{state_key}_district_change_ranking.csv'
    
    if filepath.exists():
        ranking_df = pd.read_csv(filepath)
        top = ranking_df.iloc[0]
        highest_change_districts[state_key] = {
            'name': top['district'],
            'total_change_km2': top['total_change_km2'],
            'change_pct': top['change_pct'],
            'dominant_transition': top['dominant_transition'],
        }
        print(f'{state_cfg["name"]}: Highest-change district = {top["district"]}')
        print(f'  Total change: {top["total_change_km2"]:.1f} km² ({top["change_pct"]:.1f}%)')
        print(f'  Dominant transition: {top["dominant_transition"]}')
        
        print(f'\nTop 5 districts by change:')
        display(ranking_df.head())
        print()
    else:
        print(f'⚠️ Not found: {filepath.name} — run Module 2 first')

## 3.2: Load/Compute Annual LULC Data (2016–2025)

Two options:
- **Option A**: Use Tier 1 server-side annual compositions (exported as CSVs from GEE)
- **Option B**: Use Tier 3 annual 10m raster crops (exported as GeoTIFFs)

Option A is preferred since it's computed at 10m accuracy without downloading rasters.

In [ ]:
# --- OPTION A: If you computed annual compositions server-side in GEE ---
# Load annual composition JSON for the highest-change district

annual_data = {}

for state_key, state_cfg in STATES.items():
    if state_key not in highest_change_districts:
        continue
    
    district_name = highest_change_districts[state_key]['name']
    filepath = GEE_EXPORTS_DIR / f'{state_key}_{district_name}_annual_composition.json'
    
    if filepath.exists():
        with open(filepath, 'r') as f:
            annual_hist = json.load(f)
        
        # Build annual area table
        rows = []
        for year in ALL_YEARS:
            hist = parse_histogram(annual_hist.get(str(year), {}), scale=TIER1_SCALE)
            row = {'year': year}
            for cls_val in range(NUM_CLASSES):
                row[LULC_CLASSES[cls_val]] = hist.get(cls_val, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
            rows.append(row)
        
        annual_data[state_key] = pd.DataFrame(rows)
        print(f'{state_cfg["name"]} — {district_name}: Annual data loaded ({len(ALL_YEARS)} years)')
    else:
        print(f'⚠️ Not found: {filepath.name}')
        print(f'   → You need to run annual GEE exports for district {district_name}')
        print(f'   → Or use Option B with raster crops below')

In [ ]:
# --- OPTION B: If you exported annual 10m rasters from GEE ---
# Compute pixel counts from the GeoTIFF crops

import rasterio
from config import LULC_DIR

for state_key, state_cfg in STATES.items():
    if state_key in annual_data:
        continue  # Already loaded via Option A
    if state_key not in highest_change_districts:
        continue
    
    district_name = highest_change_districts[state_key]['name']
    lulc_10m_dir = LULC_DIR.parent / 'lulc_10m_crops'
    
    rows = []
    for year in ALL_YEARS:
        tif_path = lulc_10m_dir / f'{state_key}_{district_name}_{year}.tif'
        if not tif_path.exists():
            # Try alternative naming
            tif_path = lulc_10m_dir / f'highest_change_district_{year}.tif'
        
        if tif_path.exists():
            with rasterio.open(tif_path) as src:
                data = src.read(1)
                pixel_size = abs(src.transform[0] * src.transform[4])
            
            row = {'year': year}
            for cls_val in range(NUM_CLASSES):
                count = np.sum(data == cls_val)
                row[LULC_CLASSES[cls_val]] = count * PIXEL_AREA_KM2[TIER3_SCALE]
            rows.append(row)
        else:
            print(f'  ⚠️ Missing: {tif_path.name}')
    
    if rows:
        annual_data[state_key] = pd.DataFrame(rows)
        print(f'{state_cfg["name"]} — {district_name}: {len(rows)} annual rasters loaded')

## 3.3: Time-Series Plots

In [ ]:
for state_key, state_cfg in STATES.items():
    if state_key not in annual_data:
        continue
    
    district_name = highest_change_districts[state_key]['name']
    df = annual_data[state_key]
    
    # Time-series line plot
    plot_annual_timeseries(df, district_name, state_cfg['name'])
    
    # Year-over-year change rates
    fig, ax = plt.subplots(figsize=(14, 6))
    class_names = [LULC_CLASSES[i] for i in range(NUM_CLASSES)]
    
    for cls_val in range(NUM_CLASSES):
        cls_name = LULC_CLASSES[cls_val]
        if cls_name in df.columns:
            yoy_change = df[cls_name].diff()  # Year-over-year absolute change
            ax.plot(df['year'].iloc[1:], yoy_change.iloc[1:],
                    'o-', label=cls_name, color=LULC_COLORS[cls_val],
                    linewidth=1.5, markersize=4)
    
    ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Year')
    ax.set_ylabel('Year-over-Year Change (km²)')
    ax.set_title(f'Annual LULC Change Rate — {district_name}, {state_cfg["name"]}')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    save_figure(fig, f'{state_key}_{district_name.lower()}_yoy_change.png', 'lulc_maps')
    plt.show()

print('✅ Time-series plots generated.')

## 3.4: Statistical Analysis — Gradual vs. Abrupt

In [ ]:
# For each LULC class, fit linear regression and detect change points

for state_key, state_cfg in STATES.items():
    if state_key not in annual_data:
        continue
    
    district_name = highest_change_districts[state_key]['name']
    df = annual_data[state_key]
    
    print(f'\n{"="*60}')
    print(f'Statistical Analysis — {district_name}, {state_cfg["name"]}')
    print(f'{"="*60}')
    
    results = []
    
    for cls_val in range(NUM_CLASSES):
        cls_name = LULC_CLASSES[cls_val]
        if cls_name not in df.columns:
            continue
        
        y = df[cls_name].values
        x = df['year'].values
        
        if np.std(y) < 1e-6:  # No variation
            continue
        
        # Linear regression
        slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(x, y)
        r_squared = r_value ** 2
        
        # Classify as gradual or abrupt
        # High R² → linear trend → gradual
        # Low R² → non-linear → likely abrupt changes
        pattern = 'Gradual' if r_squared > 0.7 else 'Abrupt/Non-linear'
        
        # Detect maximum year-over-year jump
        yoy = np.abs(np.diff(y))
        max_jump_idx = np.argmax(yoy)
        max_jump_year = int(x[max_jump_idx + 1])
        max_jump_val = yoy[max_jump_idx]
        
        results.append({
            'Class': cls_name,
            'Slope (km²/yr)': round(slope, 2),
            'R²': round(r_squared, 3),
            'p-value': f'{p_value:.4f}',
            'Pattern': pattern,
            'Max Jump Year': max_jump_year,
            'Max Jump (km²)': round(max_jump_val, 2),
        })
    
    results_df = pd.DataFrame(results)
    display(results_df)
    
    # Save
    save_composition_csv(
        results_df,
        PROCESSED_DIR / 'transition_matrices' / f'{state_key}_{district_name}_gradual_abrupt.csv'
    )

## 3.5: Change-Point Detection (Pettitt / CUSUM)

In [ ]:
def pettitt_test(data):
    """Simple Pettitt test for a single change-point in time series."""
    n = len(data)
    if n < 5:
        return None, 1.0
    
    # Compute U statistics
    U = np.zeros(n)
    for t in range(1, n):
        s = 0
        for i in range(t):
            for j in range(t, n):
                s += np.sign(data[j] - data[i])
        U[t] = abs(s)
    
    K = np.max(U)
    change_point = np.argmax(U)
    
    # Approximate p-value
    p_value = 2 * np.exp(-6 * K**2 / (n**3 + n**2))
    p_value = min(p_value, 1.0)
    
    return change_point, p_value


for state_key, state_cfg in STATES.items():
    if state_key not in annual_data:
        continue
    
    district_name = highest_change_districts[state_key]['name']
    df = annual_data[state_key]
    
    print(f'\nChange-Point Detection — {district_name}, {state_cfg["name"]}')
    print('-' * 50)
    
    for cls_val in range(NUM_CLASSES):
        cls_name = LULC_CLASSES[cls_val]
        if cls_name not in df.columns:
            continue
        
        y = df[cls_name].values
        if np.std(y) < 1e-6:
            continue
        
        cp_idx, p_val = pettitt_test(y)
        if cp_idx is not None:
            cp_year = int(df['year'].iloc[cp_idx])
            sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.1 else 'ns'
            print(f'  {cls_name:20s}: Change-point at {cp_year} (p={p_val:.4f}) {sig}')

## 3.6: Interpretation

Relate observed patterns to known events. Fill in the interpretation below based on results.

In [ ]:
# Summary visualization: combined gradual vs abrupt chart
for state_key, state_cfg in STATES.items():
    if state_key not in annual_data:
        continue
    
    district_name = highest_change_districts[state_key]['name']
    df = annual_data[state_key]
    
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    axes = axes.flatten()
    
    for idx, cls_val in enumerate(range(NUM_CLASSES)):
        cls_name = LULC_CLASSES[cls_val]
        ax = axes[idx]
        
        if cls_name not in df.columns:
            ax.set_visible(False)
            continue
        
        y = df[cls_name].values
        x = df['year'].values
        
        ax.plot(x, y, 'o-', color=LULC_COLORS[cls_val], linewidth=2, markersize=5)
        
        # Add linear trend line
        if np.std(y) > 1e-6:
            slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)
            trend_line = slope * x + intercept
            ax.plot(x, trend_line, '--', color='gray', alpha=0.7)
            ax.text(0.05, 0.95, f'R²={r_value**2:.3f}',
                    transform=ax.transAxes, fontsize=9, va='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_title(cls_name, fontsize=11, color=LULC_COLORS[cls_val])
        ax.set_xlabel('Year', fontsize=9)
        ax.set_ylabel('Area (km²)', fontsize=9)
        ax.grid(True, alpha=0.3)
    
    fig.suptitle(f'Annual LULC Trends — {district_name}, {state_cfg["name"]}\n'
                 f'(Dashed line = linear trend; R² shown)',
                 fontsize=14, fontweight='bold')
    fig.tight_layout()
    save_figure(fig, f'{state_key}_{district_name.lower()}_trend_grid.png', 'lulc_maps')
    plt.show()

print('\n✅ Module 3 complete.')

---
## Summary

- ✅ Identified highest-change district per state
- ✅ Annual time-series (2016–2025) line plots
- ✅ Year-over-year change rate analysis
- ✅ Linear regression + R² for gradual/abrupt classification
- ✅ Pettitt change-point detection
- ✅ 3×3 grid of class-wise trend lines with R²

**Next**: `04_water_body_size.ipynb`